# GAN Example 4 — Least Squares GAN (LSGAN)

Same DCGAN architecture as `gan_mnist.ipynb`, but with a different **loss function**. The
**LSGAN** (Mao et al., 2017) replaces the binary cross-entropy with a plain
**least-squares (MSE)** loss — a tiny change that often makes training smoother and
reduces vanishing gradients.

## Why least squares?

The standard GAN's cross-entropy loss **saturates**: once the discriminator confidently
labels a fake as fake, the gradient flowing back to the generator nearly vanishes, even
if that fake sits far from the real data. LSGAN instead penalises samples by **how far**
their score is from the target value, so even confidently-rejected fakes still get a
useful push toward the data. The losses become:

$$\mathcal{L}_D = \tfrac{1}{2}\,\mathbb{E}[(D(x)-1)^2] + \tfrac{1}{2}\,\mathbb{E}[(D(G(z)))^2], \qquad \mathcal{L}_G = \tfrac{1}{2}\,\mathbb{E}[(D(G(z))-1)^2].$$

The discriminator now outputs a **raw number** (no sigmoid) that it regresses toward `1`
for reals and `0` for fakes. **That single swap — MSE instead of cross-entropy — is the
entire difference from example 1.**

> Tested with `tensorflow==2.1`. Same cost as the DCGAN example.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers
import matplotlib.pyplot as plt

tf.random.set_seed(0)
print('TensorFlow', tf.__version__)

In [ ]:
(train_images, _), _ = tf.keras.datasets.mnist.load_data()
train_images = (train_images.reshape(-1, 28, 28, 1).astype('float32') - 127.5) / 127.5
BATCH_SIZE, NOISE_DIM = 256, 100
dataset = tf.data.Dataset.from_tensor_slices(train_images).shuffle(60000).batch(BATCH_SIZE)
print('Train:', train_images.shape)

## 1. Models (identical to the DCGAN)

The generator and discriminator are exactly as in example 1. Note the discriminator's
final layer is a single **linear** unit — LSGAN regresses it, it is not a probability.

In [ ]:
def make_generator():
    return tf.keras.Sequential([
        layers.Dense(7*7*256, use_bias=False, input_shape=(NOISE_DIM,)),
        layers.BatchNormalization(), layers.LeakyReLU(), layers.Reshape((7, 7, 256)),
        layers.Conv2DTranspose(128, 5, 1, 'same', use_bias=False),
        layers.BatchNormalization(), layers.LeakyReLU(),
        layers.Conv2DTranspose(64, 5, 2, 'same', use_bias=False),
        layers.BatchNormalization(), layers.LeakyReLU(),
        layers.Conv2DTranspose(1, 5, 2, 'same', use_bias=False, activation='tanh'),
    ])

def make_discriminator():
    return tf.keras.Sequential([
        layers.Conv2D(64, 5, 2, 'same', input_shape=[28, 28, 1]),
        layers.LeakyReLU(), layers.Dropout(0.3),
        layers.Conv2D(128, 5, 2, 'same'), layers.LeakyReLU(), layers.Dropout(0.3),
        layers.Flatten(), layers.Dense(1),   # linear output (regressed, not sigmoid)
    ])

generator = make_generator()
discriminator = make_discriminator()
g_opt = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)
d_opt = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)

## 2. The least-squares losses

This is the whole point of LSGAN — `mse` instead of binary cross-entropy.

In [ ]:
mse = tf.keras.losses.MeanSquaredError()

@tf.function
def train_step(images):
    bsz = tf.shape(images)[0]
    noise = tf.random.normal([bsz, NOISE_DIM])
    with tf.GradientTape() as gt, tf.GradientTape() as dt:
        fake = generator(noise, training=True)
        real_out = discriminator(images, training=True)
        fake_out = discriminator(fake, training=True)
        # regress real->1, fake->0
        d_loss = 0.5 * (mse(tf.ones_like(real_out), real_out) +
                        mse(tf.zeros_like(fake_out), fake_out))
        # generator wants fake->1
        g_loss = 0.5 * mse(tf.ones_like(fake_out), fake_out)
    d_opt.apply_gradients(zip(dt.gradient(d_loss, discriminator.trainable_variables),
                              discriminator.trainable_variables))
    g_opt.apply_gradients(zip(gt.gradient(g_loss, generator.trainable_variables),
                              generator.trainable_variables))
    return g_loss, d_loss

In [ ]:
seed = tf.random.normal([16, NOISE_DIM])
def plot_samples(epoch=None):
    imgs = generator(seed, training=False)
    plt.figure(figsize=(4, 4))
    for i in range(16):
        plt.subplot(4, 4, i + 1)
        plt.imshow(imgs[i, :, :, 0] * 0.5 + 0.5, cmap='gray'); plt.axis('off')
    if epoch is not None:
        plt.suptitle(f'Epoch {epoch}')
    plt.show()

## 3. Train

In [ ]:
EPOCHS = 30
for epoch in range(1, EPOCHS + 1):
    for batch in dataset:
        g_loss, d_loss = train_step(batch)
    print(f'Epoch {epoch:3d}/{EPOCHS}  gen_loss={float(g_loss):.3f}  disc_loss={float(d_loss):.3f}')
    if epoch % 5 == 0 or epoch == 1:
        plot_samples(epoch)

### Things to try

- Run example 1 (BCE) and this (MSE) side by side and compare stability.
- Change the fake/real targets (e.g. `a=-1, b=1, c=0`) per the LSGAN paper.
- Combine LSGAN loss with the conditional setup from example 2.
- Try the same loss on the 2D toy problem to visualise its effect on mode coverage.